<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup → Excel Grid Logic → V0 Backtest → Verification Report → Cash Movement / Grid Cashflow → Monthly Cashflow → Monthly Portfolio Net P&L → Audit → Results

## 1. Setup & Data

In [ ]:
import os, heapq, bisect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'

SYMBOL = 'BTCUSDT'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'

BUY_FEE = 0.001
SELL_FEE = 0.001

# KZM Excel checkpoint parameters
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

def load_market_data(symbol, timeframe, data_dir):
    path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    df['open_time'] = pd.to_datetime(df['open_time'], utc=True)
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].astype(float)

    return (
        df.drop_duplicates('open_time')
        .sort_values('open_time')
        .reset_index(drop=True)
    )

df_1m = load_market_data(SYMBOL, '1m', DATA_DIR)

start_ts = pd.Timestamp(START_DATE, tz='UTC')
end_ts = pd.Timestamp(END_DATE, tz='UTC')
df_1m = df_1m.loc[
    (df_1m['open_time'] >= start_ts)
    & (df_1m['open_time'] < end_ts)
].reset_index(drop=True)

print(
    f'Rows: {len(df_1m):,} | '
    f'{df_1m.open_time.min()} -> {df_1m.open_time.max()}'
)

## 2. Excel Grid Logic

KZM Excel is the source of truth for the fixed-grid formulas. The checkpoint below uses an independently known value from the Excel workbook.

In [ ]:
def build_excel_grid_table(
    capital,
    ceiling,
    floor,
    gap,
    buy_fee=0.001,
    sell_fee=0.001,
):
    if capital <= 0:
        raise ValueError('capital must be greater than 0.')
    if ceiling <= floor:
        raise ValueError('ceiling must be greater than floor.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')

    raw_levels = (ceiling - floor) / gap
    if not np.isclose(raw_levels, round(raw_levels)):
        raise ValueError(
            '(ceiling - floor) must be exactly divisible by gap.'
        )

    n_levels = int(round(raw_levels))
    capital_per_level = capital / n_levels

    buy_prices = ceiling - gap * np.arange(1, n_levels + 1)
    sell_prices = buy_prices + gap

    gross_base_amount = capital_per_level / buy_prices
    buy_fee_base = gross_base_amount * buy_fee
    base_amount = gross_base_amount - buy_fee_base

    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_level

    return pd.DataFrame({
        'level': np.arange(1, n_levels + 1),
        'buy_price': buy_prices,
        'sell_price': sell_prices,
        'capital_per_level': capital_per_level,
        'gross_base_amount': gross_base_amount,
        'buy_fee_base': buy_fee_base,
        'base_amount': base_amount,
        'gross_sell': gross_sell,
        'sell_fee_quote': sell_fee_quote,
        'net_sell': net_sell,
        'profit': profit,
    })

df_grid_excel = build_excel_grid_table(
    GRID_CAPITAL,
    GRID_CEILING,
    GRID_FLOOR,
    GRID_GAP,
    BUY_FEE,
    SELL_FEE,
)

excel_check = df_grid_excel.iloc[0]
EXPECTED_FIRST_PROFIT = 0.1750644398340242

assert np.isclose(
    excel_check['profit'],
    EXPECTED_FIRST_PROFIT,
    atol=1e-12,
)

print(
    f'Excel checkpoint PASSED: '
    f'{excel_check.buy_price:.0f} -> {excel_check.sell_price:.0f}, '
    f'profit={excel_check.profit:.6f}'
)

## 3. Historical Grid Configuration — V0

Capital and Gap remain strategy inputs like the Excel model. The full-period historical Low/High are used only to select V0 boundaries, so this version still has look-ahead bias and is intended for execution-engine validation.

In [ ]:
BACKTEST_CAPITAL = 3000.0
BACKTEST_GAP = 1000.0
PRICE_ROUNDING = 1000.0

historical_low = df_1m['low'].min()
historical_high = df_1m['high'].max()

BACKTEST_FLOOR = (
    np.floor(historical_low / PRICE_ROUNDING)
    * PRICE_ROUNDING
)
BACKTEST_CEILING = (
    np.ceil(historical_high / PRICE_ROUNDING)
    * PRICE_ROUNDING
)

raw_backtest_levels = (
    BACKTEST_CEILING - BACKTEST_FLOOR
) / BACKTEST_GAP

if not np.isclose(
    raw_backtest_levels,
    round(raw_backtest_levels),
):
    raise ValueError(
        'Backtest range must be divisible by BACKTEST_GAP.'
    )

NUMBER_OF_GRIDS = int(round(raw_backtest_levels))
CAPITAL_PER_LEVEL = (
    BACKTEST_CAPITAL / NUMBER_OF_GRIDS
)

df_grid_backtest = build_excel_grid_table(
    BACKTEST_CAPITAL,
    BACKTEST_CEILING,
    BACKTEST_FLOOR,
    BACKTEST_GAP,
    BUY_FEE,
    SELL_FEE,
)

print('===== V0 Backtest Grid Configuration =====')
print(f'Historical Low     : {historical_low:,.2f} USDT')
print(f'Historical High    : {historical_high:,.2f} USDT')
print(f'Grid Floor         : {BACKTEST_FLOOR:,.2f} USDT')
print(f'Grid Ceiling       : {BACKTEST_CEILING:,.2f} USDT')
print(f'Grid Gap           : {BACKTEST_GAP:,.2f} USDT')
print(f'Number of Grids    : {NUMBER_OF_GRIDS}')
print(f'Capital            : {BACKTEST_CAPITAL:,.2f} USDT')
print(f'Capital / Grid     : {CAPITAL_PER_LEVEL:,.2f} USDT')

## 4. Backtest Engine — 1 Minute

V0 execution rules:

- BUY only on a downward crossing of a grid level.
- Existing SELL targets may fill when candle High reaches the target.
- A new BUY cannot SELL in the same candle.
- A grid sold in a candle cannot rebuy in that same candle.
- Same-candle SELL proceeds are not reused for BUYs.
- BUY fee is deducted from BTC received; SELL fee is deducted from USDT proceeds, matching Excel.

In [ ]:
def run_grid_backtest(df_price, grid_table, initial_capital):
    required = {'open_time', 'open', 'high', 'low', 'close'}
    missing = required.difference(df_price.columns)

    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    if len(df_price) == 0:
        raise ValueError('df_price is empty.')
    if initial_capital <= 0:
        raise ValueError(
            'initial_capital must be greater than 0.'
        )

    data = (
        df_price.sort_values('open_time')
        .reset_index(drop=True)
    )
    grid = (
        grid_table.sort_values('buy_price')
        .reset_index(drop=True)
        .copy()
    )

    if len(grid) == 0:
        raise ValueError('grid_table is empty.')

    buy_prices = grid['buy_price'].to_numpy(float)
    sell_prices = grid['sell_price'].to_numpy(float)
    capital_per_level = (
        grid['capital_per_level'].to_numpy(float)
    )
    base_amount = grid['base_amount'].to_numpy(float)
    buy_fee_base = grid['buy_fee_base'].to_numpy(float)
    sell_fee_quote = (
        grid['sell_fee_quote'].to_numpy(float)
    )
    net_sell = grid['net_sell'].to_numpy(float)
    cycle_profit = grid['profit'].to_numpy(float)
    excel_level = grid['level'].to_numpy(int)

    if (
        len(grid) > 1
        and not np.allclose(
            np.diff(buy_prices),
            np.diff(buy_prices)[0],
        )
    ):
        raise ValueError('Arithmetic grid required.')

    holding = np.zeros(len(grid), dtype=bool)
    buy_time = [None] * len(grid)
    sell_heap = []

    cash = float(initial_capital)
    open_btc = 0.0
    realized_profit = 0.0
    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    trade_events = []
    completed_trades = []

    n_rows = len(data)
    equity_values = np.empty(n_rows)
    cash_values = np.empty(n_rows)
    btc_values = np.empty(n_rows)

    buy_price_list = buy_prices.tolist()
    prev_close = None
    event_id = 0

    for i, row in enumerate(data.itertuples(index=False)):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # 1) Existing SELL orders.
        while (
            sell_heap
            and sell_heap[0][0] <= high_price
        ):
            _, k = heapq.heappop(sell_heap)

            if not holding[k]:
                continue

            cash_before = cash
            btc_before = open_btc

            holding[k] = False
            cash += net_sell[k]
            open_btc -= base_amount[k]

            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            realized_profit += cycle_profit[k]
            total_sell_fee_usdt += sell_fee_quote[k]
            completed_cycles += 1
            sold_this_candle.add(k)
            event_id += 1

            completed_trades.append({
                'grid_level': int(excel_level[k]),
                'buy_time': buy_time[k],
                'sell_time': timestamp,
                'buy_price': buy_prices[k],
                'sell_price': sell_prices[k],
                'cost': capital_per_level[k],
                'quote_cost': capital_per_level[k],
                'base_amount': base_amount[k],
                'actual_earn': net_sell[k],
                'net_sell': net_sell[k],
                'grid_cashflow': cycle_profit[k],
                'profit': cycle_profit[k],
            })

            trade_events.append({
                'event_id': event_id,
                'time': timestamp,
                'side': 'SELL',
                'grid_level': int(excel_level[k]),
                'price': sell_prices[k],
                'base_amount': base_amount[k],
                'quote_amount': net_sell[k],
                'fee_base': 0.0,
                'fee_quote': sell_fee_quote[k],
                'realized_profit': cycle_profit[k],
                'cash_movement': net_sell[k],
                'grid_cashflow': cycle_profit[k],
                'cash_before': cash_before,
                'cash_after': cash,
                'btc_before': btc_before,
                'btc_after': open_btc,
            })

            buy_time[k] = None

        # 2) Downward BUY crossings.
        # Same-candle SELL proceeds are not available to BUYs.
        buy_budget = cash_at_candle_start
        down_start = (
            open_price
            if prev_close is None
            else max(prev_close, open_price)
        )

        if low_price < down_start:
            first_idx = bisect.bisect_left(
                buy_price_list,
                low_price,
            )
            stop_idx = bisect.bisect_left(
                buy_price_list,
                down_start,
            )

            # Higher levels are crossed first on a downward move.
            for k in range(
                stop_idx - 1,
                first_idx - 1,
                -1,
            ):
                if (
                    holding[k]
                    or k in sold_this_candle
                ):
                    continue

                cost = capital_per_level[k]

                if buy_budget + 1e-12 < cost:
                    break

                cash_before = cash
                btc_before = open_btc

                holding[k] = True
                buy_time[k] = timestamp

                buy_budget -= cost
                cash -= cost
                open_btc += base_amount[k]

                total_buy_fee_btc += buy_fee_base[k]
                total_buy_fee_usdt_equiv += (
                    buy_fee_base[k]
                    * buy_prices[k]
                )

                heapq.heappush(
                    sell_heap,
                    (sell_prices[k], k),
                )
                event_id += 1

                trade_events.append({
                    'event_id': event_id,
                    'time': timestamp,
                    'side': 'BUY',
                    'grid_level': int(excel_level[k]),
                    'price': buy_prices[k],
                    'base_amount': base_amount[k],
                    'quote_amount': cost,
                    'fee_base': buy_fee_base[k],
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_movement': -cost,
                    'grid_cashflow': 0.0,
                    'cash_before': cash_before,
                    'cash_after': cash,
                    'btc_before': btc_before,
                    'btc_after': open_btc,
                })

        # 3) Mark portfolio to market at candle Close.
        equity_values[i] = (
            cash + open_btc * close_price
        )
        cash_values[i] = cash
        btc_values[i] = open_btc
        prev_close = close_price

    equity_curve = pd.DataFrame({
        'open_time': data['open_time'].to_numpy(),
        'close': data['close'].to_numpy(float),
        'cash': cash_values,
        'btc': btc_values,
        'equity': equity_values,
    })

    running_peak = np.maximum.accumulate(
        equity_values
    )
    drawdown = (
        equity_values / running_peak - 1.0
    )
    equity_curve['drawdown'] = drawdown

    max_drawdown = float(drawdown.min())
    final_equity = float(equity_values[-1])
    net_return = (
        final_equity / initial_capital - 1.0
    )

    elapsed_days = (
        data['open_time'].iloc[-1]
        - data['open_time'].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan
    if (
        elapsed_days > 0
        and final_equity > 0
    ):
        annualized_log_growth = (
            np.log(
                final_equity / initial_capital
            )
            * (365.25 / elapsed_days)
        )
        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(annualized_log_growth)
            )

    calmar_ratio = np.nan
    if (
        max_drawdown < 0
        and np.isfinite(annualized_return)
    ):
        calmar_ratio = float(
            annualized_return
            / abs(max_drawdown)
        )

    trade_log = pd.DataFrame(trade_events)

    if not trade_log.empty:
        trade_log['cumulative_cash_movement'] = (
            trade_log['cash_movement'].cumsum()
        )
        trade_log['cumulative_grid_cashflow'] = (
            trade_log['grid_cashflow'].cumsum()
        )

    summary = {
        'initial_capital': float(initial_capital),
        'final_equity': final_equity,
        'net_return': float(net_return),
        'annualized_return': annualized_return,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'completed_cycles': int(completed_cycles),
        'open_positions': int(holding.sum()),
        'final_cash': float(cash),
        'final_btc': float(open_btc),
        'realized_profit': float(realized_profit),
        'unrealized_pnl': float(
            final_equity
            - initial_capital
            - realized_profit
        ),
        'buy_fee_btc': float(total_buy_fee_btc),
        'buy_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
        ),
        'sell_fee_usdt': float(
            total_sell_fee_usdt
        ),
        'total_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
            + total_sell_fee_usdt
        ),
    }

    return {
        'summary': summary,
        'trade_log': trade_log,
        'completed_trades': pd.DataFrame(
            completed_trades
        ),
        'equity_curve': equity_curve,
        'grid_state': grid.assign(
            holding=holding,
            buy_time=buy_time,
        ),
    }

## 5. Verification Test Report — Human-readable

This report is intentionally based on **small synthetic price paths with known expected outcomes**. Expected values are hard-coded independently of the backtest result, so the table lets us inspect `Expected` versus `Actual` directly.

The full automated Unit Test suite remains in `tests/test_grid_trading.py`.

In [ ]:
def make_verification_candles(rows):
    rows = list(rows)
    times = pd.date_range(
        '2024-01-01',
        periods=len(rows),
        freq='min',
        tz='UTC',
    )

    return pd.DataFrame({
        'open_time': times,
        'open': [r[0] for r in rows],
        'high': [r[1] for r in rows],
        'low': [r[2] for r in rows],
        'close': [r[3] for r in rows],
    })

def build_verification_report():
    report = []

    def add(
        scenario,
        expected,
        actual,
        passed,
    ):
        report.append({
            'scenario': scenario,
            'expected': expected,
            'actual': actual,
            'result': (
                'PASS'
                if passed
                else 'FAIL'
            ),
        })

    # Independent Excel oracle.
    excel_grid = build_excel_grid_table(
        3000.0,
        8987.0,
        1987.0,
        70.0,
        0.001,
        0.001,
    )
    actual_excel_profit = float(
        excel_grid.iloc[0]['profit']
    )
    expected_excel_profit = (
        0.1750644398340242
    )
    add(
        'Excel first-grid profit',
        f'{expected_excel_profit:.12f}',
        f'{actual_excel_profit:.12f}',
        np.isclose(
            actual_excel_profit,
            expected_excel_profit,
            atol=1e-12,
        ),
    )

    # Synthetic 3-grid system:
    # 100->110, 110->120, 120->130
    # Capital = 300, so Cost/Grid = 100.
    test_grid = build_excel_grid_table(
        300.0,
        130.0,
        100.0,
        10.0,
        0.001,
        0.001,
    )

    # Upward-only move must not BUY.
    up_result = run_grid_backtest(
        make_verification_candles([
            (115.0, 125.0, 115.0, 122.0),
        ]),
        test_grid,
        300.0,
    )
    up_buys = int(
        up_result['trade_log']['side']
        .eq('BUY')
        .sum()
        if len(up_result['trade_log'])
        else 0
    )
    add(
        'Upward-only candle',
        'BUY count = 0',
        f'BUY count = {up_buys}',
        up_buys == 0,
    )

    # 125 -> 115 crosses only BUY 120.
    down_result = run_grid_backtest(
        make_verification_candles([
            (125.0, 126.0, 115.0, 118.0),
        ]),
        test_grid,
        300.0,
    )
    down_log = down_result['trade_log']
    actual_buy_price = (
        float(down_log.iloc[0]['price'])
        if len(down_log) == 1
        else np.nan
    )
    add(
        'Downward 125 -> 115 crossing',
        'BUY 120 only',
        (
            f'BUY {actual_buy_price:.0f}'
            if np.isfinite(actual_buy_price)
            else 'unexpected event count'
        ),
        (
            len(down_log) == 1
            and down_log.iloc[0]['side'] == 'BUY'
            and np.isclose(
                actual_buy_price,
                120.0,
            )
        ),
    )

    add(
        'Cash after one 100-USDT BUY',
        '200.00 USDT',
        (
            f'{down_result["summary"]["final_cash"]:.2f} USDT'
        ),
        np.isclose(
            down_result['summary']['final_cash'],
            200.0,
            atol=1e-10,
        ),
    )

    # Even if High reaches 135, a position opened in
    # the same candle cannot SELL.
    same_candle_result = run_grid_backtest(
        make_verification_candles([
            (125.0, 135.0, 115.0, 120.0),
        ]),
        test_grid,
        300.0,
    )
    same_sells = int(
        same_candle_result['trade_log']['side']
        .eq('SELL')
        .sum()
    )
    add(
        'Same-candle BUY -> SELL protection',
        'SELL count = 0',
        f'SELL count = {same_sells}',
        same_sells == 0,
    )

    # Buy 120 in minute 1, then High 131 in minute 2.
    cycle_result = run_grid_backtest(
        make_verification_candles([
            (125.0, 126.0, 115.0, 118.0),
            (120.0, 131.0, 120.0, 130.0),
        ]),
        test_grid,
        300.0,
    )
    cycle_log = cycle_result['trade_log']

    actual_sides = ' -> '.join(
        cycle_log['side'].tolist()
    )
    add(
        'Later-candle completed cycle',
        'BUY -> SELL',
        actual_sides,
        cycle_log['side'].tolist()
        == ['BUY', 'SELL'],
    )

    actual_sell_price = float(
        cycle_log.iloc[-1]['price']
    )
    add(
        'Paired SELL target',
        'SELL 130',
        f'SELL {actual_sell_price:.0f}',
        np.isclose(
            actual_sell_price,
            130.0,
        ),
    )

    # Independent hand-calculated expected value:
    # Cost 100; buy at 120; 0.1% buy fee in BTC;
    # sell at 130; 0.1% sell fee in USDT.
    expected_grid_cashflow = 8.116775
    actual_grid_cashflow = float(
        cycle_result['completed_trades']
        .iloc[0]['grid_cashflow']
    )
    add(
        '120 -> 130 Grid Cashflow',
        f'{expected_grid_cashflow:.6f} USDT',
        f'{actual_grid_cashflow:.6f} USDT',
        np.isclose(
            actual_grid_cashflow,
            expected_grid_cashflow,
            atol=1e-12,
        ),
    )

    actual_open_positions = int(
        cycle_result['summary']
        ['open_positions']
    )
    add(
        'Position state after completed cycle',
        'Open positions = 0',
        (
            f'Open positions = '
            f'{actual_open_positions}'
        ),
        actual_open_positions == 0,
    )

    return pd.DataFrame(report)

df_verification_report = (
    build_verification_report()
)

display(df_verification_report)

verification_failed = (
    df_verification_report.loc[
        df_verification_report[
            'result'
        ].eq('FAIL')
    ]
)

assert verification_failed.empty, (
    'VERIFICATION REPORT FAILED:\n'
    + verification_failed.to_string(
        index=False
    )
)

print(
    'VERIFICATION REPORT: '
    'ALL SCENARIOS PASSED'
)

## 6. Run Historical Backtest

In [ ]:
backtest_result = run_grid_backtest(
    df_1m,
    df_grid_backtest,
    BACKTEST_CAPITAL,
)

backtest_summary = backtest_result['summary']
df_trade_log = backtest_result['trade_log']
df_completed_trades = (
    backtest_result['completed_trades']
)
df_equity_curve = backtest_result['equity_curve']
df_grid_state = backtest_result['grid_state']

print('Historical backtest completed.')

## 7. Cash Movement & Excel Grid Cashflow

- **Cash Movement** = actual USDT movement in/out of the cash account. BUY is negative and SELL is positive.
- **Grid Cashflow (Excel)** = `Actual Earn - Cost` for completed cycles only. This is the realized grid profit definition used in the Excel model.

In [ ]:
df_cash_ledger = df_trade_log[
    [
        'event_id',
        'time',
        'side',
        'grid_level',
        'price',
        'cash_movement',
        'cumulative_cash_movement',
        'cash_before',
        'cash_after',
        'base_amount',
        'btc_before',
        'btc_after',
        'fee_base',
        'fee_quote',
    ]
].copy()

df_grid_cashflow = df_completed_trades[
    [
        'grid_level',
        'buy_time',
        'sell_time',
        'buy_price',
        'sell_price',
        'cost',
        'actual_earn',
        'grid_cashflow',
    ]
].copy()

df_grid_cashflow[
    'cumulative_grid_cashflow'
] = (
    df_grid_cashflow[
        'grid_cashflow'
    ].cumsum()
)

net_cash_movement = (
    df_cash_ledger['cash_movement'].sum()
)

print('===== Cash Movement Reconciliation =====')
print(
    f'Initial Capital       : '
    f'{BACKTEST_CAPITAL:,.2f} USDT'
)
print(
    f'Net Cash Movement     : '
    f'{net_cash_movement:,.2f} USDT'
)
print(
    f'Expected Final Cash   : '
    f'{BACKTEST_CAPITAL + net_cash_movement:,.2f} USDT'
)
print(
    f'Backtest Final Cash   : '
    f'{backtest_summary["final_cash"]:,.2f} USDT'
)

print(
    '\n===== Excel Grid Cashflow '
    'Reconciliation ====='
)
print(
    f'Completed Cycles      : '
    f'{len(df_grid_cashflow):,}'
)
print(
    f'Total Grid Cashflow   : '
    f'{df_grid_cashflow.grid_cashflow.sum():,.2f} USDT'
)
print(
    f'Realized Profit       : '
    f'{backtest_summary["realized_profit"]:,.2f} USDT'
)

display(df_grid_cashflow.head(20))
display(df_cash_ledger.head(20))

## 8. Monthly Cumulative Grid Cashflow

This uses **Excel Grid Cashflow (`Actual Earn - Cost`)** grouped by the month in which the SELL completes the cycle. Zero-trade months are retained.

In [ ]:
months = pd.period_range(
    pd.Period(START_DATE, freq='M'),
    (
        pd.Timestamp(END_DATE)
        - pd.Timedelta(days=1)
    ).to_period('M'),
    freq='M',
)

grid_cashflow_by_month = (
    df_grid_cashflow.assign(
        month=pd.to_datetime(
            df_grid_cashflow['sell_time'],
            utc=True,
        ).dt.tz_localize(None).dt.to_period('M')
    )
)

monthly_grid_cashflow = (
    grid_cashflow_by_month
    .groupby('month')['grid_cashflow']
    .sum()
    .reindex(months, fill_value=0.0)
)

monthly_completed_cycles = (
    grid_cashflow_by_month
    .groupby('month')
    .size()
    .reindex(months, fill_value=0)
)

df_grid_cashflow_monthly = pd.DataFrame({
    'month': months.astype(str),
    'monthly_grid_cashflow': (
        monthly_grid_cashflow.to_numpy(float)
    ),
    'completed_cycles': (
        monthly_completed_cycles.to_numpy(int)
    ),
})

df_grid_cashflow_monthly[
    'cumulative_grid_cashflow'
] = (
    df_grid_cashflow_monthly[
        'monthly_grid_cashflow'
    ].cumsum()
)

assert np.isclose(
    df_grid_cashflow_monthly[
        'cumulative_grid_cashflow'
    ].iloc[-1],
    backtest_summary['realized_profit'],
    atol=1e-8,
)

display(df_grid_cashflow_monthly)

plt.figure(figsize=(14, 5))
plt.plot(
    df_grid_cashflow_monthly['month'],
    df_grid_cashflow_monthly[
        'cumulative_grid_cashflow'
    ],
    marker='o',
)
plt.title(
    'BTC Spot Fixed Grid — '
    'Cumulative Grid Cashflow by Month'
)
plt.xlabel('Month')
plt.ylabel(
    'Cumulative Grid Cashflow (USDT)'
)
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Monthly Portfolio Net P&L

**Portfolio Net P&L** is defined as the change in **month-end total portfolio equity**:

`Month-end Cash + BTC × Month-end BTC Close`

Therefore it includes both realized and unrealized mark-to-market P&L, with trading fees already reflected in the portfolio balances.

- First month: `Month-end Equity - Initial Capital`
- Later months: `Current Month-end Equity - Previous Month-end Equity`
- The sum of monthly Net P&L must equal `Final Equity - Initial Capital`.

In [ ]:
def build_monthly_portfolio_pnl(
    equity_curve,
    initial_capital,
    start_date=None,
    end_date=None,
):
    if len(equity_curve) == 0:
        raise ValueError('equity_curve is empty.')
    if initial_capital <= 0:
        raise ValueError(
            'initial_capital must be greater than 0.'
        )

    df = equity_curve[
        ['open_time', 'equity']
    ].copy()

    df['open_time'] = pd.to_datetime(
        df['open_time'],
        utc=True,
    )
    df = df.sort_values('open_time')

    df['month'] = (
        df['open_time']
        .dt.tz_localize(None)
        .dt.to_period('M')
    )

    month_end_equity = (
        df.groupby('month')['equity']
        .last()
    )

    if start_date is not None:
        first_month = pd.Period(
            start_date,
            freq='M',
        )
    else:
        first_month = (
            month_end_equity.index.min()
        )

    if end_date is not None:
        last_month = (
            pd.Timestamp(end_date)
            - pd.Timedelta(days=1)
        ).to_period('M')
    else:
        last_month = (
            month_end_equity.index.max()
        )

    month_index = pd.period_range(
        first_month,
        last_month,
        freq='M',
    )

    # A missing full month has no valid month-end
    # mark-to-market price; fail instead of silently
    # fabricating a portfolio value.
    missing_months = (
        month_index
        .difference(month_end_equity.index)
    )
    if len(missing_months):
        raise ValueError(
            'Missing month-end equity for: '
            + ', '.join(
                missing_months.astype(str)
            )
        )

    month_end_equity = (
        month_end_equity.reindex(month_index)
    )

    beginning_equity = (
        month_end_equity.shift(1)
    )
    beginning_equity.iloc[0] = (
        float(initial_capital)
    )

    portfolio_net_pnl = (
        month_end_equity
        - beginning_equity
    )

    monthly_return = (
        portfolio_net_pnl
        / beginning_equity
    )

    result = pd.DataFrame({
        'month': month_index.astype(str),
        'beginning_equity': (
            beginning_equity.to_numpy(float)
        ),
        'ending_equity': (
            month_end_equity.to_numpy(float)
        ),
        'portfolio_net_pnl': (
            portfolio_net_pnl.to_numpy(float)
        ),
        'monthly_return': (
            monthly_return.to_numpy(float)
        ),
    })

    result['cumulative_portfolio_pnl'] = (
        result['ending_equity']
        - float(initial_capital)
    )

    return result

df_portfolio_pnl_monthly = (
    build_monthly_portfolio_pnl(
        equity_curve=df_equity_curve,
        initial_capital=BACKTEST_CAPITAL,
        start_date=START_DATE,
        end_date=END_DATE,
    )
)

portfolio_gain = (
    backtest_summary['final_equity']
    - BACKTEST_CAPITAL
)

assert np.isclose(
    df_portfolio_pnl_monthly[
        'portfolio_net_pnl'
    ].sum(),
    portfolio_gain,
    atol=1e-8,
)

display(df_portfolio_pnl_monthly)

plt.figure(figsize=(14, 5))
plt.bar(
    df_portfolio_pnl_monthly['month'],
    df_portfolio_pnl_monthly[
        'portfolio_net_pnl'
    ],
)
plt.axhline(0)
plt.title(
    'BTC Spot Fixed Grid — '
    'Monthly Portfolio Net P&L'
)
plt.xlabel('Month')
plt.ylabel('Net P&L (USDT)')
plt.xticks(rotation=90)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. System Audit

These are full-run reconciliation checks on the historical backtest. They are separate from both the human-readable Verification Report and the synthetic Unit Test suite.

In [ ]:
system_tests = []

def add_system_test(
    name,
    passed,
    expected,
    actual,
):
    system_tests.append({
        'test': name,
        'status': (
            'PASS'
            if passed
            else 'FAIL'
        ),
        'expected': expected,
        'actual': actual,
    })

add_system_test(
    'Grid spacing = configured Gap',
    bool(np.allclose(
        df_grid_backtest['sell_price']
        - df_grid_backtest['buy_price'],
        BACKTEST_GAP,
    )),
    BACKTEST_GAP,
    'all levels',
)

cash_identity_error = (
    df_trade_log['cash_before']
    + df_trade_log['cash_movement']
    - df_trade_log['cash_after']
).abs().max()

add_system_test(
    'Cash movement identity',
    cash_identity_error <= 1e-9,
    '<= 1e-9',
    cash_identity_error,
)

net_cash_movement = (
    df_trade_log['cash_movement'].sum()
)
expected_final_cash = (
    BACKTEST_CAPITAL
    + net_cash_movement
)

add_system_test(
    'Cash reconciliation',
    abs(
        expected_final_cash
        - backtest_summary['final_cash']
    ) <= 1e-8,
    expected_final_cash,
    backtest_summary['final_cash'],
)

total_grid_cashflow = (
    df_trade_log['grid_cashflow'].sum()
)

add_system_test(
    'Grid Cashflow = realized profit',
    abs(
        total_grid_cashflow
        - backtest_summary['realized_profit']
    ) <= 1e-8,
    backtest_summary['realized_profit'],
    total_grid_cashflow,
)

equity_identity_error = (
    df_equity_curve['cash']
    + df_equity_curve['btc']
    * df_equity_curve['close']
    - df_equity_curve['equity']
).abs().max()

add_system_test(
    'Cash + BTC x Close = Equity',
    equity_identity_error <= 1e-8,
    '<= 1e-8',
    equity_identity_error,
)

add_system_test(
    'Cash never negative',
    df_equity_curve['cash'].min()
    >= -1e-9,
    '>= 0',
    df_equity_curve['cash'].min(),
)

add_system_test(
    (
        'Monthly cumulative Grid Cashflow '
        '= realized profit'
    ),
    abs(
        df_grid_cashflow_monthly[
            'cumulative_grid_cashflow'
        ].iloc[-1]
        - backtest_summary['realized_profit']
    ) <= 1e-8,
    backtest_summary['realized_profit'],
    df_grid_cashflow_monthly[
        'cumulative_grid_cashflow'
    ].iloc[-1],
)

monthly_portfolio_pnl_sum = (
    df_portfolio_pnl_monthly[
        'portfolio_net_pnl'
    ].sum()
)
total_portfolio_gain = (
    backtest_summary['final_equity']
    - backtest_summary['initial_capital']
)

add_system_test(
    (
        'Sum Monthly Portfolio Net P&L '
        '= total portfolio gain'
    ),
    abs(
        monthly_portfolio_pnl_sum
        - total_portfolio_gain
    ) <= 1e-8,
    total_portfolio_gain,
    monthly_portfolio_pnl_sum,
)

add_system_test(
    (
        'Final cumulative Portfolio P&L '
        '= total portfolio gain'
    ),
    abs(
        df_portfolio_pnl_monthly[
            'cumulative_portfolio_pnl'
        ].iloc[-1]
        - total_portfolio_gain
    ) <= 1e-8,
    total_portfolio_gain,
    df_portfolio_pnl_monthly[
        'cumulative_portfolio_pnl'
    ].iloc[-1],
)

df_system_test_log = pd.DataFrame(
    system_tests
)

display(df_system_test_log)

failed_system_tests = (
    df_system_test_log.loc[
        df_system_test_log[
            'status'
        ].eq('FAIL')
    ]
)

assert failed_system_tests.empty, (
    'SYSTEM AUDIT FAILED:\n'
    + failed_system_tests.to_string(
        index=False
    )
)

print(
    'SYSTEM AUDIT: '
    'ALL TESTS PASSED'
)

## 11. Results

In [ ]:
s = backtest_summary

print('===== V0 Backtest Summary =====')
print(
    f'Initial Capital     : '
    f'{s["initial_capital"]:,.2f} USDT'
)
print(
    f'Final Equity        : '
    f'{s["final_equity"]:,.2f} USDT'
)
print(
    f'Net Return          : '
    f'{s["net_return"]:.2%}'
)
print(
    f'Annualized Return   : '
    f'{s["annualized_return"]:.2%}'
)
print(
    f'Max Drawdown        : '
    f'{s["max_drawdown"]:.2%}'
)
print(
    f'Calmar Ratio        : '
    f'{s["calmar_ratio"]:.3f}'
)
print(
    f'Completed Cycles    : '
    f'{s["completed_cycles"]:,}'
)
print(
    f'Open Positions      : '
    f'{s["open_positions"]:,}'
)
print(
    f'Final Cash          : '
    f'{s["final_cash"]:,.2f} USDT'
)
print(
    f'Final BTC           : '
    f'{s["final_btc"]:.8f} BTC'
)
print(
    f'Realized Profit     : '
    f'{s["realized_profit"]:,.2f} USDT'
)
print(
    f'Unrealized P&L      : '
    f'{s["unrealized_pnl"]:,.2f} USDT'
)
print(
    f'Total Fee (USDT eq.): '
    f'{s["total_fee_usdt_equiv"]:,.2f} USDT'
)

daily_equity = (
    df_equity_curve
    .set_index('open_time')['equity']
    .resample('1D')
    .last()
    .dropna()
)

plt.figure(figsize=(14, 5))
plt.plot(
    daily_equity.index,
    daily_equity.values,
)
plt.title(
    'BTC Spot Fixed Grid — '
    'Daily Portfolio Equity'
)
plt.xlabel('Date')
plt.ylabel('Equity (USDT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()